<a href="https://colab.research.google.com/github/jjmoncus/MV_coursework_2/blob/main/Machine_Vision_Final_Lab_Model_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Dependencies

In [3]:
# ===== INSTALL DEPENDENCIES =====
!pip install huggingface_hub
!pip install boto3 -q
!pip install opencv-python torch numpy torchvision tqdm
!pip install mediapipe==0.10.13

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.13 (from versions: 0.10.30, 0.10.31)
ERROR: No matching distribution found for mediapipe==0.10.13


In [5]:
# Import the required libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import cv2
import numpy as np
from tqdm import tqdm
import time

import mediapipe as mp

import warnings

# Suppress Protobuf and MediaPipe internal warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf.symbol_database')

# Find and assign device

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


# Please double, triple, quadruple check that the below code runs without errors before submitting.

## TODO 1 - Enter your HuggingFace username below:

In [7]:

hf_username = "jjmoncus1"

## TODO 2 - Define your model EXACTLY as you did in your training code (otherwise there will be errors, and, possibly, tears).

Note below the classname is 'YourModelArchitecture'. That's because it literally needs to be YOUR MODEL ARCHITECTURE. This class definition is later referred to below in the 'load_model_from_hub' method. The architecture must match here, or it will not be able to instantiate the model weights correctly once it downloads them from HuggingFace. Pay very close attention to getting this right, please.

Replace the below code, and replace the corresponding line in the 'load_model_from_hub' method.

In [ ]:
class PoseModel(nn.Module):
    def __init__(self, num_joints=33, num_features=4, num_classes=10):
        super().__init__()
        
        # Total input channels = joints * features (e.g., 33 * 4 = 132)
        self.in_channels = num_joints * num_features


        self.input_bn = nn.BatchNorm1d(self.in_channels)

        # initial FC for compressing the joint/feature information
        self.fc_1 = nn.Sequential(

            nn.Linear(self.in_channels, 64),
            nn.LeakyReLU(0.1)
        )


        # process the joint/feature info (channels) over time
        self.temporal_conv = nn.Sequential(
            
            # Layer 1: [B, 132, 100] ---> [B, 64, 50]
            nn.Conv1d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.1),

            # Layer 2: [B, 64, 50] ---> [B, 128, 25]
            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.1),

            # Layer 3: [B, 128, 25] ---> [B, 256, 13]
            nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),

            # Global Average Pooling: [B, 256, 13] ---> [B, 256, 1]
            nn.AdaptiveAvgPool1d(1)
        )
        
        # Output of temporal is [B, 256, 1]. Flattened = 256
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 112),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            nn.Linear(112, num_classes),
        )

    def forward(self, x):
        """
        Input x: [Batch, Time, Joints, Features]
        """
        
        B, T, J, F = x.shape

        # flatten joints and features into channel dimension
        x = x.view(B, T, J * F)       # [B, T, 132]

        # batch norm before first linear layer
        x = x.transpose(1, 2)          # [B, 132, T]
        x = self.input_bn(x)           # Normalize across the batch/time
        x = x.transpose(1, 2)          # Back to [B, T, 132]

        # send through FC_1
        x = self.fc_1(x)              # [B, T, 32]

        # move dimension
        x = x.transpose(1, 2)         # [B, 32, T]
        
        # send through temporal layers
        x = self.temporal_conv(x)     # [B, 256, 1]
        
        # classify
        return self.classifier(x)     # [B, 10, 1]


## Download the test data from s3, and create the corresponding dataset + dataloader.

There's no TODO for you here. This text is just here to explain to you what this code does.

In this instance, the test data IS the training data you were provided in the Model Training notebook. This is by design. You do not have access to the test data. This is a simple check to make sure the mechanics of this notebook work.

You should achieve the same accuracy here in this notebook, as you did in your previous notebook (random seed notwithstanding).

In [ ]:
# # =============================================================================
# # DOWNLOAD TEST DATA FROM S3
# # =============================================================================

# def download_test_data(bucket_name='training-and-validation-data',download_dir='./test-data'):
#     s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

#     bucket_name = 'prism-mvta'
#     prefix = 'training-and-validation-data/'

#     os.makedirs(download_dir, exist_ok=True)

#     paginator = s3.get_paginator('list_objects_v2')
#     pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

#     video_names = []

#     for page in pages:
#         if 'Contents' not in page:
#             print("No files found at the specified path!")
#             break

#         print("Downloading test data:\n")
#         for obj in tqdm(page['Contents']):
#             key = obj['Key']
#             filename = os.path.basename(key)

#             if not filename:
#                 continue

#             video_names.append(filename)
#             local_path = os.path.join(download_dir, filename)
#             # print(f"Downloading: {filename}")
#             s3.download_file(bucket_name, key, local_path)

#     print(f"\nDownloaded {len(video_names)} test videos")
#     return download_dir
    

# Run Test Data Through Pose Esimation

Function for extracting mediapope joint coordinates

In [ ]:
# def extract_mediapipe_tensor(video_tensor):
#     """
#     Input: video_tensor of shape (C, T, H, W)
#     Output: result_tensor of shape (T, 33, 4)
#     """
#     C, T, H, W = video_tensor.shape
    
#     # Pre-allocate output tensor (T frames, 33 landmarks, 4 features: x, y, z, vis)
#     output_data = torch.zeros((T, 33, 4))

#     for t in range(T):
#         # 1. Slice the frame: (C, H, W)
#         frame = video_tensor[:, t, :, :]
        
#         # 2. Permute to (H, W, C) and convert to Numpy
#         # Note: If your tensor is BGR, use frame[[2, 1, 0], :, :].permute...
#         frame_np = frame.permute(1, 2, 0).cpu().numpy()
        
#         # 3. Ensure uint8 format (MediaPipe requires 0-255)
#         if frame_np.dtype != np.uint8:
#             if frame_np.max() <= 1.01:
#                 frame_np = (frame_np * 255).astype(np.uint8)
#             else:
#                 frame_np = frame_np.astype(np.uint8)

#         # 4. Process frame
#         results = pose.process(frame_np)

#         # 5. Fill landmark data if a person is detected
#         if results.pose_landmarks:
#             for i, lm in enumerate(results.pose_landmarks.landmark):
#                 output_data[t, i, 0] = lm.x
#                 output_data[t, i, 1] = lm.y
#                 output_data[t, i, 2] = lm.z
#                 output_data[t, i, 3] = lm.visibility
#         else:
#             # If no detection, the frame remains zeros
#             pass
            
#     return output_data

Function for batch extracting poses from a bunch of videos

In [8]:
def export_poses(video_paths, search_dir, output_dir):
    """
    Iterates through video_paths, extracts pose data, and saves as .pt tensors.
    """
    print("\nEstimating Poses in Training Videos...\n")
    
    # Refresh output directory exists
    
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"{output_dir} already exists - refreshing now.")
    else:
        print("f{output_dir} not found - creating.")
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize MediaPipe once
    mp_pose = mp.solutions.pose
    pose_processor = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

    # Wrap the loop with tqdm for a progress bar
    for video_name in tqdm(video_paths, desc="Processing Videos", unit="video"):
        # 2. Construct full input path
        video_input_path = os.path.join(search_dir, video_name)
        
        if not os.path.exists(video_input_path):
            # Using tqdm.write prevents the progress bar from breaking
            tqdm.write(f"⚠️ Skip: {video_name} not found in {search_dir}")
            continue

        # 3. Extract landmarks using OpenCV loop
        cap = cv2.VideoCapture(video_input_path)
        all_landmarks = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose_processor.process(frame_rgb)
            
            current_frame = np.zeros((33, 4))
            if results.pose_world_landmarks:
                for i, lm in enumerate(results.pose_world_landmarks.landmark):
                    current_frame[i] = [lm.x, lm.y, lm.z, lm.visibility]
            
            all_landmarks.append(current_frame)
        
        cap.release()

        # 4. Convert to Torch Tensor [T, 33, 4]
        video_tensor = torch.from_numpy(np.array(all_landmarks)).float()

        # 5. Save to Output Directory
        base_name = os.path.splitext(video_name)[0]
        output_filename = f"{base_name}_pose.pt"
        save_path = os.path.join(output_dir, output_filename)
        
        torch.save(video_tensor, save_path)
        
        # Uncommenting the official "saved" write statement for now
        # tqdm.write(f"✅ Saved: {output_filename}")

# Data Loaders

In [9]:

class PoseDataset(Dataset):
    """Dataset for loading poses from a folder. Labels from filename prefix."""

    def __init__(self, pose_dir):
        self.pose_dir = pose_dir
        
        self.video_files = [ f for f in os.listdir(pose_dir)if f.endswith(('.pt')) ]

        self.labels = [ int(f.split('_')[0]) for f in self.video_files]

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = os.path.join(self.pose_dir, self.video_files[idx])
        frames = self._load_video(video_path)
        label = self.labels[idx]

        return frames, label

    def _load_video(self, path, stride = 10, max_frames = 100):
        
        video = torch.load(path)
        frames = []
        frame_count = 0
        T = video.shape[0]

        for t in range(T):
            if t % stride == 0:
                frame = video[t, :, :]
                frames.append(frame)
            if len(frames) == max_frames:
                break

        # frames = torch.from_numpy(np.array(frames)).permute(3, 0, 1, 2).float() / 255
        frames = torch.from_numpy(np.array(frames))

        return frames


def collate_fn(batch):
    frames_list, labels = zip(*batch)
    target_frames = 100 

    padded_frames = []
    
    # For each video, ...
    for frames in frames_list:
        
        # frames shape: (T, 33, 4)
        num_frames = frames.shape[0]
        
        # if the video is too short, ...
        if num_frames < target_frames:

            # ... grab the last frame: (C, 1, H, W)
            last_frame = frames[-1:, :, :]
            
            # ... calculate how many times to repeat it
            padding_size = target_frames - num_frames
            
            # ... create the padding by repeating the last frame
            padding = last_frame.repeat(padding_size, 1, 1)
            
            # ..., and concatenate along the T dimension (0)
            frames = torch.cat([frames, padding], dim=0)
        
        # if video is too long ...
        elif num_frames > target_frames:
            
            # ... truncate to 100 frames
            frames = frames[:target_frames, :, :]
        
        # Finally, add our padded video to the list of videos
        padded_frames.append(frames)

    # Combine list into a batch tensor: (Batch, T, 33, 4)
    frames_batch = torch.stack(padded_frames, dim=0)
    labels_batch = torch.tensor(labels)

    return frames_batch, labels_batch

## TODO 3 - Download your model from HuggingFace and instantiate it

Replace line 8 of the below code. Line 8 is where you instantiate YOUR MODEL ARCHITECTURE (which you re-defined above) with the weights you download from HuggingFace. Make sure you get the class name, and the arguments to the __init__ method correct.


This code just downloads the same model which you uploaded in the last notebook.

In [9]:
# =============================================================================
# DOWNLOAD MODEL FROM HUGGING FACE
# =============================================================================

def load_model_from_hub(repo_id):
    model_path = hf_hub_download(repo_id=repo_id, filename="model.pt")

    model = PoseModel()
    model.load_state_dict(torch.load(model_path, map_location='cpu'))

    print(f"Model loaded from {repo_id}")
    return model

model = load_model_from_hub(f"{hf_username}/mv-final-assignment")

Model loaded from jjmoncus1/mv-final-assignment


## TODO 4

Make sure the below code correctly evaluates your model performance on the given data!

This is your last chance to verify this before submission.

In [10]:
def evaluate(model, test_loader, dataset, device):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_times = []

    print("\n")

    with torch.no_grad():
        for idx, (frames, labels) in enumerate(test_loader):
            frames, labels = frames.to(device), labels.to(device)

            # Time the forward pass
            start_time = time.time()
            outputs = model(frames)
            if device.type == 'cuda':
                torch.cuda.synchronize()  # wait for GPU to finish
            end_time = time.time()

            inference_time = (end_time - start_time) * 1000  # ms
            all_times.append(inference_time)

            _, preds = torch.max(outputs, 1)
            preds = preds + 1

            for i in range(labels.size(0)):
                batch_idx = idx * test_loader.batch_size + i
                video_name = dataset.video_files[batch_idx]
                pred = preds[i].item()
                true_label = labels[i].item()
                is_correct = "✓" if pred == true_label else "✗"

                print(f"{is_correct}  pred={pred}  true={true_label}  |  {inference_time:>7.1f}ms  |  {video_name}")

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return accuracy, all_preds, all_labels, all_times


# =============================================================================
# RUN INFERENCE
# =============================================================================

def run_inference(model, bucket_name='training-and-validation-data'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    ### NEED TO UNCOMMENT THE ACTUAL DOWNLOAD TEST DATA SHEET
    # Download test data
    # test_dir = download_test_data(bucket_name, './test-data')
    test_dir = "./extra-data"

    # perform pose estimation on test-data, output to test-pose-data
    video_paths = [f for f in os.listdir(test_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
    export_poses(video_paths, test_dir, "extra-pose-data")

    model = model.to(device)

    # Create dataloader
    test_dataset = PoseDataset("extra-pose-data")
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    print(f"\nRunning inference on {len(test_dataset)} test videos...")

    # Warmup (optional, helps get consistent GPU timings)
    if device.type == 'cuda':
        dummy = torch.randn(1, 3, 100, 224, 224).to(device)
        with torch.no_grad():
            _ = model(dummy)
        torch.cuda.synchronize()

    total_start = time.time()
    accuracy, preds, labels, times = evaluate(model, test_loader, test_dataset, device)
    total_end = time.time()

    # Summary
    num_correct = sum(p == l for p, l in zip(preds, labels))
    num_wrong = len(preds) - num_correct

    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total videos:         {len(preds)}")
    print(f"Correct:              {num_correct}")
    print(f"Incorrect:                {num_wrong}")
    print(f"")
    print(f"ACCURACY:             {accuracy*100:.2f}%")
    print(f"")
    print(f"Total time:           {total_end - total_start:.2f}s")
    print(f"Avg per video:        {sum(times) / len(times):.1f}ms")
    print(f"Min latency:          {min(times):.1f}ms")
    print(f"Max latency:          {max(times):.1f}ms")
    print("="*50)
    return accuracy, preds, labels

_, _, _ = run_inference(model)

NameError: name 'model' is not defined

In [15]:
model = model.to(device)

# Create dataloader
test_dataset = PoseDataset("test-pose-data")
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print(f"\nRunning inference on {len(test_dataset)} test videos...")

# Warmup (optional, helps get consistent GPU timings)
if device.type == 'cuda':
    dummy = torch.randn(1, 3, 100, 224, 224).to(device)
    with torch.no_grad():
        _ = model(dummy)
    torch.cuda.synchronize()



Running inference on 0 test videos...


In [ ]:

total_start = time.time()
accuracy, preds, labels, times = evaluate(model, test_loader, test_dataset, device)
total_end = time.time()

# Summary
num_correct = sum(p == l for p, l in zip(preds, labels))
num_wrong = len(preds) - num_correct

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Total videos:         {len(preds)}")
print(f"Correct:              {num_correct}")
print(f"Incorrect:                {num_wrong}")
print(f"")
print(f"ACCURACY:             {accuracy*100:.2f}%")
print(f"")
print(f"Total time:           {total_end - total_start:.2f}s")
print(f"Avg per video:        {sum(times) / len(times):.1f}ms")
print(f"Min latency:          {min(times):.1f}ms")
print(f"Max latency:          {max(times):.1f}ms")
print("="*50)
return accuracy, preds, labels